In [ ]:
import torch
from torch import nn

import torch_pruning as tp

from snp import utils, importances, pruning, gram_matrices

In [ ]:
model_name = 'vgg16'
batch_size = 128
seed = 42
curvature = 'sse'

data_path = 'PATH/TO/DATA/FOLDER'

In [5]:
importance_score = 'act'            # CHOOSE YOUR DESIRED IMPORTANCE SCORE HERE

match importance_score:
    case 'tso':
        reconstruct = True
        eps = 0.
        use_bias = True
        normalize = False
    case 'act':
        reconstruct = True
        eps = 0.
        use_bias = True
        normalize = True
    case 'saw':
        reconstruct = False
        eps = 0.
        use_bias = False
        normalize = False
    case 'pfa':
        reconstruct = False
        eps = 1e-3
        use_bias = False
        normalize = True


In [ ]:
train_data, _ = utils.load_data(data_path, batch_size=32, n_train_samples=256)
model = utils.load_model(model_name).cuda()

utils.assign_names(model)

In [ ]:
if importance_score == 'pfa':
    means = gram_matrices.compute_means(model, train_data)
    means = {k: means[k].cuda() for k in means.keys()}
else:
    means = None

gram_matrices = gram_matrices.compute_gram_matrices(model, train_data, means)

In [ ]:
model = utils.load_model(model_name).cuda()
_, test_data = utils.load_data(data_path, 256)

utils.assign_names(model)
utils.set_gram_matrices(model, gram_matrices)

example_input = torch.randn(1, 3, 224, 224).cuda()
utils.set_num_patches(model, example_input)

ignored_layers = [model.classifier[-1]]

DG = tp.DependencyGraph().build_dependency(
    model,
    example_inputs=example_input,
    ignored_layers=[],
    ignored_params=[],
)


local_scores, calibrated_scores = importances.compute_importance_scores(
    DG, ignored_layers, importance_score, importances.first_layer, use_bias, eps=eps, normalize=normalize
)

unit_orders_per_group = [
    torch.argsort(scores, descending=True) for scores in local_scores
]

In [ ]:
for ratio in torch.linspace(0.05, 0.9, 18):
    model = utils.load_model(model_name).cuda()
    utils.assign_names(model)
    utils.set_num_patches(model, example_input)

    utils.set_gram_matrices(model, gram_matrices)

    DG = tp.DependencyGraph().build_dependency(
        model,
        example_inputs=example_input,
        ignored_layers=[],
        ignored_params=[],
    )
    ignored_layers = [model.classifier[-1]]

    units_pruned_per_group, n_total_pruned = pruning.compute_pruned_nodes(
        local_scores, calibrated_scores, ratio
    )

    pruning.prune_model(
        DG,
        ignored_layers,
        units_pruned_per_group,
        unit_orders_per_group,
        use_bias,
        eps=eps,
        reconstruct=reconstruct
    )

    flop, pc = tp.utils.count_ops_and_params(model, example_input)
    loss, acc = utils.measure_perf(model, nn.CrossEntropyLoss(), test_data, "cuda")

    print(ratio, flop, pc, loss, acc)


100%|██████████| 196/196 [02:17<00:00,  1.42it/s]


tensor(0.0500) 13570655706.0 124708638 1.1723090642547607 0.70912


100%|██████████| 196/196 [02:18<00:00,  1.41it/s]


tensor(0.1000) 12099016593.0 111500048 1.234842314682007 0.69416


100%|██████████| 196/196 [02:03<00:00,  1.59it/s]


tensor(0.1500) 10841253007.0 99419325 1.3239458893585205 0.6764


... (26 outputs left out)

100%|██████████| 196/196 [01:11<00:00,  2.73it/s]


tensor(0.8500) 495449695.0 3993092 7.036209899749756 0.001


100%|██████████| 196/196 [01:11<00:00,  2.73it/s]

tensor(0.9000) 251229387.0 1886307 7.068277018890381 0.001
